In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torch transformers datasets scikit-learn tqdm pandas

In [ ]:
# ====================================================
# STEP A — Extract DistilBERT Embeddings & Save Them
# ====================================================

!pip install transformers -q

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import DistilBertTokenizerFast, DistilBertModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

CSV_PATH = "/content/drive/MyDrive/phase1_train_clean.csv"
TEXT_COL = "clean_text"

ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

SAVE_EMB_PATH = "/content/drive/MyDrive/phase1_embeddings_64.pt"

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=[TEXT_COL])

print("Rows:", len(df))

# -----------------------------
# Load tokenizer + model
# -----------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
bert.eval()

# -----------------------------
# Encode → CLS embeddings
# -----------------------------
EMBS = []
BATCH_SIZE = 64

def encode_batch(texts):
    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        out = bert(**tokens).last_hidden_state[:,0,:]   # CLS embedding
    return out.cpu()  # store on CPU

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_texts = df[TEXT_COL].iloc[i:i+BATCH_SIZE].tolist()
    emb = encode_batch(batch_texts)
    EMBS.append(emb)

EMBS = torch.cat(EMBS, dim=0)   # shape: (N, 768)

# -----------------------------
# Save embeddings
# -----------------------------
labels_dict = {a: df[a].values for a in ATTRS}

torch.save({
    "embeddings": EMBS,
    "labels": labels_dict,
    "text": df[TEXT_COL].tolist(),
    "indices": df.index.tolist()
}, SAVE_EMB_PATH)

print("Saved embeddings to:", SAVE_EMB_PATH)
print("Embedding shape:", EMBS.shape)


Device: cuda
Rows: 443499


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

  0%|          | 0/6930 [00:00<?, ?it/s]

Saved embeddings to: /content/drive/MyDrive/phase1_embeddings_64.pt
Embedding shape: torch.Size([443499, 768])


In [ ]:
##phase 2


In [ ]:
# ====================================================
# STEP A — Extract DistilBERT Embeddings & Save Them
# ====================================================

!pip install transformers -q

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import DistilBertTokenizerFast, DistilBertModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

CSV_PATH = "/content/drive/MyDrive/phase2_train_clean.csv"
TEXT_COL = "clean_text"

ATTRS = [
    "supergroup", "group", "module", "brand"
]

SAVE_EMB_PATH = "/content/drive/MyDrive/phase2_embeddings.pt"

# -----------------------------
# Load data
# -----------------------------2
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=[TEXT_COL])

print("Rows:", len(df))

# -----------------------------
# Load tokenizer + model
# -----------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
bert.eval()

# -----------------------------
# Encode → CLS embeddings
# -----------------------------
EMBS = []
BATCH_SIZE = 64

def encode_batch(texts):
    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        out = bert(**tokens).last_hidden_state[:,0,:]   # CLS embedding
    return out.cpu()  # store on CPU

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_texts = df[TEXT_COL].iloc[i:i+BATCH_SIZE].tolist()
    emb = encode_batch(batch_texts)
    EMBS.append(emb)

EMBS = torch.cat(EMBS, dim=0)   # shape: (N, 768)

# -----------------------------
# Save embeddings
# -----------------------------
labels_dict = {a: df[a].values for a in ATTRS}

torch.save({
    "embeddings": EMBS,
    "labels": labels_dict,
    "text": df[TEXT_COL].tolist(),
    "indices": df.index.tolist()
}, SAVE_EMB_PATH)

print("Saved embeddings to:", SAVE_EMB_PATH)
print("Embedding shape:", EMBS.shape)


Device: cuda
Rows: 561838


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

  0%|          | 0/8779 [00:00<?, ?it/s]

Saved embeddings to: /content/drive/MyDrive/phase2_embeddings.pt
Embedding shape: torch.Size([561838, 768])


In [ ]:
# ====================================================
# STEP A — Extract DistilBERT Embeddings & Save Them
# ====================================================

!pip install transformers -q

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import DistilBertTokenizerFast, DistilBertModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

CSV_PATH = "/content/drive/MyDrive/phase1_val_clean(1).csv"
TEXT_COL = "clean_text"

ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

SAVE_EMB_PATH = "/content/drive/MyDrive/phase1_val_embeddings.pt"

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=[TEXT_COL])

print("Rows:", len(df))

# -----------------------------
# Load tokenizer + model
# -----------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
bert.eval()

# -----------------------------
# Encode → CLS embeddings
# -----------------------------
EMBS = []
BATCH_SIZE = 64

def encode_batch(texts):
    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        out = bert(**tokens).last_hidden_state[:,0,:]   # CLS embedding
    return out.cpu()  # store on CPU

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_texts = df[TEXT_COL].iloc[i:i+BATCH_SIZE].tolist()
    emb = encode_batch(batch_texts)
    EMBS.append(emb)

EMBS = torch.cat(EMBS, dim=0)   # shape: (N, 768)

# -----------------------------
# Save embeddings
# -----------------------------
labels_dict = {a: df[a].values for a in ATTRS}

torch.save({
    "embeddings": EMBS,
    "labels": labels_dict,
    "text": df[TEXT_COL].tolist(),
    "indices": df.index.tolist()
}, SAVE_EMB_PATH)

print("Saved embeddings to:", SAVE_EMB_PATH)
print("Embedding shape:", EMBS.shape)


Device: cuda
Rows: 95035


  0%|          | 0/1485 [00:00<?, ?it/s]

Saved embeddings to: /content/drive/MyDrive/phase1_val_embeddings.pt
Embedding shape: torch.Size([95035, 768])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, r2_score
)

SAVE_EMB_PATH = "/content/drive/MyDrive/phase1_embeddings.pt"
SAVE_MODEL_PATH = "/content/drive/MyDrive/multitask_classifier.pt"

ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# Load embeddings
# -----------------------------
data = torch.load(SAVE_EMB_PATH, weights_only=False)
X = data["embeddings"]             # tensor (N, 768)
text_list = data["text"]
labels_raw = data["labels"]

# -----------------------------
# Encode labels
# -----------------------------
encoders = {}
labels = {}

for a in ATTRS:
    le = LabelEncoder()
    labels[a] = le.fit_transform(labels_raw[a])
    encoders[a] = le

# Convert to tensors
Y = {a: torch.tensor(labels[a], dtype=torch.long) for a in ATTRS}

# -----------------------------
# Train/val split
# -----------------------------
train_idx, val_idx = train_test_split(
    np.arange(len(X)), test_size=0.15, random_state=42, shuffle=True
)

X_train = X[train_idx].to(device)
X_val   = X[val_idx].to(device)

Y_train = {a: Y[a][train_idx].to(device) for a in ATTRS}
Y_val   = {a: Y[a][val_idx].to(device) for a in ATTRS}

# -----------------------------
# Multi-task classifier
# -----------------------------
class MultiTaskHead(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.dropout = nn.Dropout(0.15)

        self.heads = nn.ModuleDict({
            attr: nn.Sequential(
                nn.Linear(768, 256),
                nn.ReLU(),
                nn.Dropout(0.15),
                nn.Linear(256, num_classes[attr])
            )
            for attr in ATTRS
        })

    def forward(self, x):
        x = self.dropout(x)
        return {a: self.heads[a](x) for a in ATTRS}

num_classes = {a: len(encoders[a].classes_) for a in ATTRS}
model = MultiTaskHead(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

BATCH = 256
EPOCHS = 25

# -----------------------------
# Training loop
# -----------------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    idx = torch.randperm(len(X_train))
    X_train_shuf = X_train[idx]
    Y_train_shuf = {a: Y_train[a][idx] for a in ATTRS}

    for i in range(0, len(X_train), BATCH):
        xb = X_train_shuf[i:i+BATCH]
        loss = 0

        optimizer.zero_grad()
        logits = model(xb)

        for a in ATTRS:
            yb = Y_train_shuf[a][i:i+BATCH]
            loss += criterion(logits[a], yb)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss={total_loss:.4f}")

    model.eval()
preds = {a: [] for a in ATTRS}
trues = {a: [] for a in ATTRS}

with torch.no_grad():
    for i in range(0, len(X_val), BATCH):
        xb = X_val[i:i+BATCH]
        log = model(xb)

        for a in ATTRS:
            preds[a].extend(log[a].argmax(dim=1).cpu().numpy())
            trues[a].extend(Y_val[a][i:i+BATCH].cpu().numpy())

print("\nValidation Metrics:")
for a in ATTRS:
    y_true = np.array(trues[a])
    y_pred = np.array(preds[a])

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="weighted")
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)

    # R^2 score (not meaningful for classification, but included)
    r2 = r2_score(y_true, y_pred)

    print(f"\n--- {a} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"R² Score : {r2:.4f}")

# -----------------------------
# Save classifier & encoders
# -----------------------------
torch.save({
    "state_dict": model.state_dict(),
    "encoders": {a: encoders[a].classes_.tolist() for a in ATTRS},
    "num_classes": num_classes
}, SAVE_MODEL_PATH)

Device: cuda
Epoch 1/25 | Loss=27660.9945
Epoch 2/25 | Loss=15560.2277
Epoch 3/25 | Loss=11701.3494
Epoch 4/25 | Loss=9828.3523
Epoch 5/25 | Loss=8679.4224
Epoch 6/25 | Loss=7901.4171
Epoch 7/25 | Loss=7313.5050
Epoch 8/25 | Loss=6866.6931
Epoch 9/25 | Loss=6502.4384
Epoch 10/25 | Loss=6208.8904
Epoch 11/25 | Loss=5956.2747
Epoch 12/25 | Loss=5744.9625
Epoch 13/25 | Loss=5560.3009
Epoch 14/25 | Loss=5396.1719
Epoch 15/25 | Loss=5250.8924
Epoch 16/25 | Loss=5127.6926
Epoch 17/25 | Loss=5013.8890
Epoch 18/25 | Loss=4911.7557
Epoch 19/25 | Loss=4820.7305
Epoch 20/25 | Loss=4729.3014
Epoch 21/25 | Loss=4651.1839
Epoch 22/25 | Loss=4575.1162
Epoch 23/25 | Loss=4512.9848
Epoch 24/25 | Loss=4452.0875
Epoch 25/25 | Loss=4398.0729

Validation Metrics:

--- details_Brand ---
Accuracy : 0.9556
Precision: 0.9595
Recall   : 0.9556
F1-score : 0.9540
R² Score : 0.8970

--- L0_category ---
Accuracy : 0.9172
Precision: 0.9167
Recall   : 0.9172
F1-score : 0.9162
R² Score : 0.7878

--- L1_category ---
Ac

In [ ]:
##### shared

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, r2_score
)

SAVE_EMB_PATH = "/content/drive/MyDrive/phase1_embeddings.pt"
SAVE_MODEL_PATH = "/content/drive/MyDrive/multitask_classifier_shared.pt"

ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---------------------------------------------------
# LOAD EMBEDDINGS
# ---------------------------------------------------
data = torch.load(SAVE_EMB_PATH, weights_only=False)
X = data["embeddings"]             # (N, 768)
text_list = data["text"]
labels_raw = data["labels"]

# ---------------------------------------------------
# LABEL ENCODING
# ---------------------------------------------------
encoders = {}
labels = {}

for a in ATTRS:
    le = LabelEncoder()
    labels[a] = le.fit_transform(labels_raw[a])
    encoders[a] = le

Y = {a: torch.tensor(labels[a], dtype=torch.long) for a in ATTRS}

# ---------------------------------------------------
# TRAIN/VAL SPLIT
# ---------------------------------------------------
train_idx, val_idx = train_test_split(
    np.arange(len(X)), test_size=0.15, random_state=42, shuffle=True
)

X_train = X[train_idx].to(device)
X_val   = X[val_idx].to(device)

Y_train = {a: Y[a][train_idx].to(device) for a in ATTRS}
Y_val   = {a: Y[a][val_idx].to(device) for a in ATTRS}

# ---------------------------------------------------
# MULTI-TASK CLASSIFIER WITH SHARED MLP
# ---------------------------------------------------
class MultiTaskShared(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # SHARED MLP
        self.shared = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.15)
        )

        # SEPARATE OUTPUT HEADS
        self.heads = nn.ModuleDict({
            attr: nn.Linear(256, num_classes[attr])
            for attr in ATTRS
        })

    def forward(self, x):
        h = self.shared(x)               # Shared representation
        return {a: self.heads[a](h) for a in ATTRS}

num_classes = {a: len(encoders[a].classes_) for a in ATTRS}
model = MultiTaskShared(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

BATCH = 256
EPOCHS = 30

# ---------------------------------------------------
# TRAINING LOOP
# ---------------------------------------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    idx = torch.randperm(len(X_train))
    X_train_shuf = X_train[idx]
    Y_train_shuf = {a: Y_train[a][idx] for a in ATTRS}

    for i in range(0, len(X_train), BATCH):
        xb = X_train_shuf[i:i+BATCH]
        loss = 0

        optimizer.zero_grad()
        logits = model(xb)

        for a in ATTRS:
            yb = Y_train_shuf[a][i:i+BATCH]
            loss += criterion(logits[a], yb)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss={total_loss:.4f}")

# ---------------------------------------------------
# VALIDATION
# ---------------------------------------------------
model.eval()
preds = {a: [] for a in ATTRS}
trues = {a: [] for a in ATTRS}

with torch.no_grad():
    for i in range(0, len(X_val), BATCH):
        xb = X_val[i:i+BATCH]
        out = model(xb)

        for a in ATTRS:
            preds[a].extend(out[a].argmax(dim=1).cpu().numpy())
            trues[a].extend(Y_val[a][i:i+BATCH].cpu().numpy())

print("\nValidation Metrics:")
for a in ATTRS:
    y_true = np.array(trues[a])
    y_pred = np.array(preds[a])

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted")
    r2   = r2_score(y_true, y_pred)

    print(f"\n--- {a} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"R² Score : {r2:.4f}")

# ---------------------------------------------------
# SAVE MODEL
# ---------------------------------------------------
torch.save({
    "state_dict": model.state_dict(),
    "encoders": {a: encoders[a].classes_.tolist() for a in ATTRS},
    "num_classes": num_classes
}, SAVE_MODEL_PATH)

print("\nSaved shared multi-task classifier to:", SAVE_MODEL_PATH)


Device: cuda
Epoch 1/30 | Loss=26964.0226
Epoch 2/30 | Loss=15269.4073
Epoch 3/30 | Loss=11674.2135
Epoch 4/30 | Loss=9826.9859
Epoch 5/30 | Loss=8605.5306
Epoch 6/30 | Loss=7727.7469
Epoch 7/30 | Loss=7050.2569
Epoch 8/30 | Loss=6507.9813
Epoch 9/30 | Loss=6063.2413
Epoch 10/30 | Loss=5688.3914
Epoch 11/30 | Loss=5369.6225
Epoch 12/30 | Loss=5094.8578
Epoch 13/30 | Loss=4854.8509
Epoch 14/30 | Loss=4645.8349
Epoch 15/30 | Loss=4462.0508
Epoch 16/30 | Loss=4300.1281
Epoch 17/30 | Loss=4147.9544
Epoch 18/30 | Loss=4010.2793
Epoch 19/30 | Loss=3889.8900
Epoch 20/30 | Loss=3781.1082
Epoch 21/30 | Loss=3678.1226
Epoch 22/30 | Loss=3580.9895
Epoch 23/30 | Loss=3492.2504
Epoch 24/30 | Loss=3406.8446
Epoch 25/30 | Loss=3330.8660
Epoch 26/30 | Loss=3264.4424
Epoch 27/30 | Loss=3194.5392
Epoch 28/30 | Loss=3131.2819
Epoch 29/30 | Loss=3070.7434
Epoch 30/30 | Loss=3017.0363

Validation Metrics:

--- details_Brand ---
Accuracy : 0.9424
Precision: 0.9461
Recall   : 0.9424
F1-score : 0.9402
R² Scor

In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/phase1_train_clean.csv"
df = pd.read_csv(CSV_PATH)

# Build parent → child mappings
parent_to_child = {
    "L0": {},
    "L1": {},
    "L2": {},
    "L3": {}
}

# L0 → L1
for p, c in zip(df["L0_category"], df["L1_category"]):
    parent_to_child["L0"].setdefault(p, set()).add(c)

# L1 → L2
for p, c in zip(df["L1_category"], df["L2_category"]):
    parent_to_child["L1"].setdefault(p, set()).add(c)

# L2 → L3
for p, c in zip(df["L2_category"], df["L3_category"]):
    parent_to_child["L2"].setdefault(p, set()).add(c)

# L3 → L4
for p, c in zip(df["L3_category"], df["L4_category"]):
    parent_to_child["L3"].setdefault(p, set()).add(c)

print("Hierarchy maps built!")


Hierarchy maps built!


In [ ]:
import torch
import torch.nn as nn
import numpy as np

SAVE_MODEL_PATH = "/content/drive/MyDrive/multitask_classifier_shared.pt"

ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load checkpoint
ckpt = torch.load(SAVE_MODEL_PATH, map_location=device)

# Rebuild encoders
from sklearn.preprocessing import LabelEncoder
encoders = {}
for a in ATTRS:
    le = LabelEncoder()
    le.classes_ = np.array(ckpt["encoders"][a])
    encoders[a] = le

num_classes = ckpt["num_classes"]


In [ ]:
# ---------------------------------------------------
# Multi-Task Classifier with SHARED MLP (Inference Version)
# ---------------------------------------------------
class MultiTaskShared(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Shared representation used by ALL tasks
        self.shared = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.15)
        )

        # Separate output heads
        self.heads = nn.ModuleDict({
            attr: nn.Linear(256, num_classes[attr])
            for attr in ATTRS
        })

    def forward(self, x):
        h = self.shared(x)                    # Shared features
        return {a: self.heads[a](h) for a in ATTRS}   # Task logits


# ---------------------------------------------------
# LOAD MODEL FOR INFERENCE
# ---------------------------------------------------
model = MultiTaskShared(num_classes).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()


MultiTaskShared(
  (shared): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.15, inplace=False)
  )
  (heads): ModuleDict(
    (details_Brand): Linear(in_features=256, out_features=5066, bias=True)
    (L0_category): Linear(in_features=256, out_features=27, bias=True)
    (L1_category): Linear(in_features=256, out_features=163, bias=True)
    (L2_category): Linear(in_features=256, out_features=612, bias=True)
    (L3_category): Linear(in_features=256, out_features=1252, bias=True)
    (L4_category): Linear(in_features=256, out_features=962, bias=True)
  )
)

In [ ]:
import torch.nn.functional as F

def masked_cascade_inference(embedding):
    """
    embedding: tensor of shape (768,) or (1,768)
    returns: dict with predictions for all 6 attributes
    """
    embedding = embedding.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(embedding)

    # Step 1: predict L0 directly
    L0_idx = logits["L0_category"].argmax(dim=1).item()
    L0 = encoders["L0_category"].inverse_transform([L0_idx])[0]

    # Step 2: L1 → mask invalid
    valid_L1 = list(parent_to_child["L0"][L0])
    valid_L1_idx = encoders["L1_category"].transform(valid_L1)

    mask = torch.full_like(logits["L1_category"], -1e9)
    mask[:, valid_L1_idx] = 0
    L1_idx = (logits["L1_category"] + mask).argmax(dim=1).item()
    L1 = encoders["L1_category"].inverse_transform([L1_idx])[0]

    # Step 3: L2 → mask
    valid_L2 = list(parent_to_child["L1"][L1])
    valid_L2_idx = encoders["L2_category"].transform(valid_L2)

    mask = torch.full_like(logits["L2_category"], -1e9)
    mask[:, valid_L2_idx] = 0
    L2_idx = (logits["L2_category"] + mask).argmax(dim=1).item()
    L2 = encoders["L2_category"].inverse_transform([L2_idx])[0]

    # Step 4: L3 → mask
    valid_L3 = list(parent_to_child["L2"][L2])
    valid_L3_idx = encoders["L3_category"].transform(valid_L3)

    mask = torch.full_like(logits["L3_category"], -1e9)
    mask[:, valid_L3_idx] = 0
    L3_idx = (logits["L3_category"] + mask).argmax(dim=1).item()
    L3 = encoders["L3_category"].inverse_transform([L3_idx])[0]

    # Step 5: L4 → mask
    valid_L4 = list(parent_to_child["L3"][L3])
    valid_L4_idx = encoders["L4_category"].transform(valid_L4)

    mask = torch.full_like(logits["L4_category"], -1e9)
    mask[:, valid_L4_idx] = 0
    L4_idx = (logits["L4_category"] + mask).argmax(dim=1).item()
    L4 = encoders["L4_category"].inverse_transform([L4_idx])[0]

    # Brand does not depend on hierarchy
    brand_idx = logits["details_Brand"].argmax(dim=1).item()
    brand = encoders["details_Brand"].inverse_transform([brand_idx])[0]

    return {
        "details_Brand": brand,
        "L0_category": L0,
        "L1_category": L1,
        "L2_category": L2,
        "L3_category": L3,
        "L4_category": L4
    }


Validation

In [ ]:
import pandas as pd

VAL_CSV = "/content/drive/MyDrive/phase1_val_clean(1).csv"
df_val = pd.read_csv(VAL_CSV)

print("Rows:", len(df_val))
df_val.head()


Rows: 95035


,indoml_id,clean_text,details_Brand,L0_category,L1_category,L2_category,L3_category,L4_category
0,0,pendleton eco wise washable wool blanket black...,Pendleton,Home & Kitchen,Bedding,Blankets & Throws,Bed Blankets,na
1,1,jp london md3a049 dm1578 space nebula removabl...,JP London,Tools & Home Improvement,"Paint, Wall Treatments & Supplies",Wall Stickers & Murals,na,na
2,2,lawn fawn lf2938 fangtastic friends lawn cuts ...,Lawn Fawn,"Arts, Crafts & Sewing",Scrapbooking & Stamping,Die-Cutting & Embossing,Die-Cuts,na
3,3,ancheer foldable elliptical machine home use e...,ANCHEER,Sports & Outdoors,Exercise & Fitness,Cardio Training,Elliptical Trainers,na
4,4,schecter jeff loomis jlv 7 nt left handed 7 st...,Schecter,Musical Instruments,Guitars,Electric Guitars,Solid Body,na


In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
bert.eval()

def embed_text(text):
    tokens = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=32,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = bert(**tokens).last_hidden_state[:, 0, :]
    return emb.squeeze(0).cpu()   # shape (768,)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
val_embeddings = []

for t in df_val["clean_text"]:
    val_embeddings.append(embed_text(t))

val_embeddings = torch.stack(val_embeddings)
print(val_embeddings.shape)


torch.Size([95035, 768])


In [ ]:
Y_val = {}
for a in ATTRS:
    Y_val[a] = encoders[a].transform(df_val[a].astype(str).values)


In [ ]:
def evaluate_masked_inference_from_text():
    preds = {a: [] for a in ATTRS}
    trues = {a: [] for a in ATTRS}

    for i in range(len(val_embeddings)):
        emb = val_embeddings[i]

        result = masked_cascade_inference(emb)

        for a in ATTRS:
            pred_idx = encoders[a].transform([result[a]])[0]
            preds[a].append(pred_idx)
            trues[a].append(Y_val[a][i])

    return preds, trues


In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, r2_score
)

preds, trues = evaluate_masked_inference_from_text()

print("\n=== HIERARCHICAL MASKED INFERENCE METRICS ===")
for a in ATTRS:
    y_true = np.array(trues[a])
    y_pred = np.array(preds[a])

    print(f"\n--- {a} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"R² Score : {r2_score(y_true, y_pred):.4f}")


NameError: name 'val_embeddings' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
CSV_PATH         = "/content/drive/MyDrive/phase1_train_clean.csv"
VAL_EMB_PATH     = "/content/drive/MyDrive/phase1_val_embeddings.pt"
MODEL_PATH       = "/content/drive/MyDrive/multitask_classifier_shared.pt"

ATTRS = ["details_Brand","L0_category","L1_category","L2_category","L3_category","L4_category"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --------------------------------------------------
# LOAD HIERARCHY FROM CSV
# --------------------------------------------------
df = pd.read_csv(CSV_PATH)

parent_to_child = {lvl:{} for lvl in ["L0_category","L1_category","L2_category","L3_category"]}

def build_map(parent_col, child_col, key):
    for p, c in zip(df[parent_col], df[child_col]):
        parent_to_child[key].setdefault(p, set()).add(c)

build_map("L0_category","L1_category","L0_category")
build_map("L1_category","L2_category","L1_category")
build_map("L2_category","L3_category","L2_category")
build_map("L3_category","L4_category","L3_category")

print("Hierarchy loaded.")

# --------------------------------------------------
# LOAD MODEL + ENCODERS
# --------------------------------------------------
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

# rebuild encoders
encoders = {}
for a in ATTRS:
    le = LabelEncoder()
    le.classes_ = np.array(ckpt["encoders"][a])
    encoders[a] = le

num_classes = ckpt["num_classes"]  # dict

# shared MLP model used during training
class MultiTaskShared(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.15)
        )
        self.heads = nn.ModuleDict({
            a: nn.Linear(256, num_classes[a])
            for a in ATTRS
        })

    def forward(self, x):
        h = self.shared(x)
        return {a: self.heads[a](h) for a in ATTRS}

model = MultiTaskShared(num_classes).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

print("Model loaded.")

# --------------------------------------------------
# LOAD PRECOMPUTED VALIDATION EMBEDDINGS
# --------------------------------------------------
val_data = torch.load(VAL_EMB_PATH, map_location="cpu", weights_only=False)
val_embeddings = val_data["embeddings"]      # (N_val, 768)
val_indices    = val_data["indices"]         # original row indices
val_labels_raw = val_data["labels"]          # dictionary of raw labels

# convert labels to encoded ints using the same encoders
Y_val = {}
for a in ATTRS:
    Y_val[a] = encoders[a].transform(val_labels_raw[a])

Y_val = {a: torch.tensor(Y_val[a], dtype=torch.long) for a in ATTRS}

print("Validation embeddings loaded:", val_embeddings.shape)

# --------------------------------------------------
# BUILD PARENT→CHILD MASK MATRICES FOR FAST GPU MASKING
# --------------------------------------------------
parent_child_mask = {}

def build_mask(parent_lvl, child_lvl):
    n_parent = num_classes[parent_lvl]
    n_child  = num_classes[child_lvl]
    mask = torch.zeros((n_parent, n_child), dtype=torch.float32)

    for parent_name, children in parent_to_child[parent_lvl].items():
        if len(children) == 0:
            continue
        p_idx = encoders[parent_lvl].transform([parent_name])[0]
        c_idx = encoders[child_lvl].transform(list(children))
        mask[p_idx, c_idx] = 1.0
    parent_child_mask[child_lvl] = mask.to(device)

build_mask("L0_category","L1_category")
build_mask("L1_category","L2_category")
build_mask("L2_category","L3_category")
build_mask("L3_category","L4_category")

print("Hierarchy masks ready.")

# --------------------------------------------------
# MASKED CASCADED INFERENCE (BATCHED)
# --------------------------------------------------
def masked_cascade_inference_batch(local_logits, encoders, parent_child_mask):
    batch_size = local_logits["L0_category"].size(0)
    preds = {}

    # Brand (independent)
    preds["details_Brand"] = local_logits["details_Brand"].argmax(dim=1).cpu().tolist()

    # L0 direct
    L0_idx = local_logits["L0_category"].argmax(dim=1)
    preds["L0_category"] = L0_idx.cpu().tolist()

    # L1 masked
    def cascade(parent_idx, child_lvl, logits_child):
        mask_matrix = parent_child_mask[child_lvl]
        per_mask = mask_matrix[parent_idx]             # (batch, n_child)
        probs = F.softmax(logits_child, dim=1)
        masked_probs = probs * per_mask
        zero_mask = (per_mask.sum(dim=1) == 0)
        out = torch.zeros(batch_size, dtype=torch.long, device=device)

        if (~zero_mask).any():
            idx = (~zero_mask).nonzero(as_tuple=False).squeeze(1)
            out[idx] = masked_probs[idx].argmax(dim=1)
        if zero_mask.any():
            idx0 = zero_mask.nonzero(as_tuple=False).squeeze(1)
            out[idx0] = logits_child[idx0].argmax(dim=1)
        return out

    L1_idx = cascade(L0_idx, "L1_category", local_logits["L1_category"])
    preds["L1_category"] = L1_idx.cpu().tolist()

    L2_idx = cascade(L1_idx, "L2_category", local_logits["L2_category"])
    preds["L2_category"] = L2_idx.cpu().tolist()

    # Fix: Pass local_logits["L3_category"] to the cascade for L3_idx
    L3_idx = cascade(L2_idx, "L3_category", local_logits["L3_category"])
    preds["L3_category"] = L3_idx.cpu().tolist()

    L4_idx = cascade(L3_idx, "L4_category", local_logits["L4_category"])
    preds["L4_category"] = L4_idx.cpu().tolist()

    return preds

# --------------------------------------------------
# RUN INFERENCE + METRICS
# --------------------------------------------------
preds = {a: [] for a in ATTRS}
trues = {a: [] for a in ATTRS}

BATCH = 256
model.eval()

with torch.no_grad():
    for i in range(0, len(val_embeddings), BATCH):
        xb = val_embeddings[i:i+BATCH].to(device)
        local_logits = model(xb)

        casc = masked_cascade_inference_batch(local_logits, encoders, parent_child_mask)

        for a in ATTRS:
            preds[a].extend(casc[a])
            trues[a].extend(Y_val[a][i:i+len(casc[a])].cpu().tolist())

print("\n=== HIERARCHICAL INFERENCE METRICS ===")

for a in ATTRS:
    y_true = np.array(trues[a])
    y_pred = np.array(preds[a])

    print(f"\n--- {a} ---")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, average="weighted", zero_division=0))
    print("F1-score :", f1_score(y_true, y_pred, average="weighted"))


Device: cuda
Hierarchy loaded.
Model loaded.
Validation embeddings loaded: torch.Size([95035, 768])
Hierarchy masks ready.

=== HIERARCHICAL INFERENCE METRICS ===

--- details_Brand ---
Accuracy : 0.9422212868943021
Precision: 0.9447019281057929
Recall   : 0.9422212868943021
F1-score : 0.9402165848765435

--- L0_category ---
Accuracy : 0.9244067974956595
Precision: 0.9239957493708293
Recall   : 0.9244067974956595
F1-score : 0.9236743021434675

--- L1_category ---
Accuracy : 0.881285842058189
Precision: 0.8802517256828519
Recall   : 0.881285842058189
F1-score : 0.878366540392543

--- L2_category ---
Accuracy : 0.8476666491292681
Precision: 0.8457521984668611
Recall   : 0.8476666491292681
F1-score : 0.8418692064343543

--- L3_category ---
Accuracy : 0.8151312674277897
Precision: 0.8149980187303534
Recall   : 0.8151312674277897
F1-score : 0.8078158029727888

--- L4_category ---
Accuracy : 0.8664281580470353
Precision: 0.8670614461412802
Recall   : 0.8664281580470353
F1-score : 0.861655222

In [ ]:
print("Model heads class counts:")
for a in ATTRS:
    print(a, "→ model:", num_classes[a])

print("\nMask matrices child counts:")
for child_lvl in parent_child_mask:
    print(child_lvl, "→ mask children:", parent_child_mask[child_lvl].shape[1])


Model heads class counts:
details_Brand → model: 5066
L0_category → model: 27
L1_category → model: 163
L2_category → model: 612
L3_category → model: 1252
L4_category → model: 962

Mask matrices child counts:
L1_category → mask children: 163
L2_category → mask children: 612
L3_category → mask children: 1252
L4_category → mask children: 962


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import f1_score

# ============================================
# HMCN-Lite with:
#  ✔ Input reuse
#  ✔ Global + Local heads
#  ✔ Local+Global joint loss
#  ✔ Hierarchical violation penalty (λ)
#  ✔ β-weighted score fusion (β)
# ============================================

β = 0.6      # fusion weight
λ = 0.4      # hierarchy violation penalty weight

class HMCN_Lite(nn.Module):
    def __init__(self, num_classes, num_global):
        super().__init__()

        # Dropout
        self.dropout = nn.Dropout(0.15)

        # --------------------------------------------
        # GLOBAL NETWORK (3 layers with input reuse)
        # --------------------------------------------
        self.G1 = nn.Sequential(
            nn.Linear(768, 384),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.G2 = nn.Sequential(
            nn.Linear(768 + 384, 384),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.G3 = nn.Sequential(
            nn.Linear(768 + 384, 384),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --------------------------------------------
        # LOCAL HEADS (one for each level)
        # --------------------------------------------
        self.local_heads = nn.ModuleDict({
            attr: nn.Linear(384, num_classes[attr])
            for attr in ATTRS
        })

        # --------------------------------------------
        # GLOBAL HEAD (predicts full path)
        # --------------------------------------------
        self.global_head = nn.Linear(384, num_global)

    def forward(self, x):
        x = self.dropout(x)

        # Global hierarchy flow
        g1 = self.G1(x)
        g2 = self.G2(torch.cat([x, g1], dim=1))
        g3 = self.G3(torch.cat([x, g2], dim=1))

        # Local logits
        local_logits = {
            "details_Brand": self.local_heads["details_Brand"](g1),
            "L0_category": self.local_heads["L0_category"](g1),

            "L1_category": self.local_heads["L1_category"](g2),
            "L2_category": self.local_heads["L2_category"](g2),

            "L3_category": self.local_heads["L3_category"](g3),
            "L4_category": self.local_heads["L4_category"](g3),
        }

        # Global logits
        global_logits = self.global_head(g3)

        return local_logits, global_logits


In [ ]:
ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

# Add your full-path global label
ATTRS_GLOBAL = ATTRS + ["global_path"]
df_train = pd.read_csv("/content/drive/MyDrive/phase1_train_clean.csv")

df_train["full_path"] = df_train[["L0_category","L1_category","L2_category","L3_category","L4_category"]].agg(" || ".join, axis=1)

from sklearn.preprocessing import LabelEncoder
path_encoder = LabelEncoder()
path_ids = path_encoder.fit_transform(df_train["full_path"])

Y["global_path"] = torch.tensor(path_ids, dtype=torch.long)
Y_train["global_path"] = Y["global_path"][train_idx].to(device)
Y_val["global_path"]   = Y["global_path"][val_idx].to(device)

num_global = len(path_encoder.classes_)


In [ ]:
df_train = pd.read_csv("/content/drive/MyDrive/phase1_train_clean.csv")

parent_to_child = {
    "L0_category": {},
    "L1_category": {},
    "L2_category": {},
    "L3_category": {}
}

# Build mapping for each level L0->L1, L1->L2, L2->L3, L3->L4
for _, row in df_train.iterrows():
    # L0 → L1
    p = row["L0_category"]
    c = row["L1_category"]
    parent_to_child["L0_category"].setdefault(p, set()).add(c)

    # L1 → L2
    p = row["L1_category"]
    c = row["L2_category"]
    parent_to_child["L1_category"].setdefault(p, set()).add(c)

    # L2 → L3
    p = row["L2_category"]
    c = row["L3_category"]
    parent_to_child["L2_category"].setdefault(p, set()).add(c)

    # L3 → L4
    p = row["L3_category"]
    c = row["L4_category"]
    parent_to_child["L3_category"].setdefault(p, set()).add(c)

# Convert sets → lists
for lvl in parent_to_child:
    for p in parent_to_child[lvl]:
        parent_to_child[lvl][p] = list(parent_to_child[lvl][p])

print("Mapping built!")


Mapping built!


In [ ]:
# --- run once before training ---
# `path_encoder` is the LabelEncoder used to encode full_path during setup
# `df_train["full_path"]` exists and path_encoder.classes_ aligns with indices in global logits

# Build a matrix M_level of shape (num_global_paths, num_level_classes)
# such that global_probs @ M_level -> (batch, num_level_classes)
# M_level[path_idx, level_class_idx] = 1 if path[path_idx] has that class at that level

import numpy as np
from collections import defaultdict

# get list of full_path strings in order of path_encoder.classes_
paths = path_encoder.classes_.tolist()  # e.g. "L0 || L1 || L2 || L3 || L4"
num_global = len(paths)

# parse paths into components
split_paths = [p.split(" || ") for p in paths]  # list of [L0, L1, L2, L3, L4]

# For each level, build a mapping class -> index in encoder for that level
level_names = ["L0_category", "L1_category", "L2_category", "L3_category", "L4_category"]
level_classes = {lvl: sorted(df_train[lvl].unique()) for lvl in level_names}
level_to_index = {
    lvl: {cls: idx for idx, cls in enumerate(level_classes[lvl])}
    for lvl in level_names
}
num_level_classes = {lvl: len(level_classes[lvl]) for lvl in level_names}

# Build binary mapping matrices (torch tensors) for efficient matmul
# M_l shape: (num_global, num_level_classes[l])
import torch

M_level = {}
for lvl_idx, lvl in enumerate(level_names):
    num_cls = num_level_classes[lvl]
    M = torch.zeros((num_global, num_cls), dtype=torch.float32)
    for path_idx, comps in enumerate(split_paths):
        cls = comps[lvl_idx]
        cls_idx = level_to_index[lvl][cls]
        M[path_idx, cls_idx] = 1.0
    # move to device
    M_level[lvl] = M.to(device)

# You will use M_level["L2_category"], etc.
# Also keep level_classes and level_to_index for inverse transforms if needed.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import f1_score
import numpy as np
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score
)


# ==========================================================
# 1. BUILD GLOBAL→LEVEL MAPPING MATRICES (for β-FUSION)
# ==========================================================

# list of full paths in correct encoded order
paths = path_encoder.classes_.tolist()
num_global = len(paths)

# split "L0 || L1 || L2 || L3 || L4"
split_paths = [p.split(" || ") for p in paths]

# level names
level_names = ["L0_category", "L1_category", "L2_category", "L3_category", "L4_category"]

# class lists per level
level_classes = {lvl: sorted(df_train[lvl].unique()) for lvl in level_names}

# mapping from class label → index
level_to_index = {
    lvl: {cls: idx for idx, cls in enumerate(level_classes[lvl])}
    for lvl in level_names
}

# number of classes per level
num_level_classes = {lvl: len(level_classes[lvl]) for lvl in level_names}

# Build binary mapping matrices (num_global paths → classes in that level)
M_level = {}
for lvl_idx, lvl in enumerate(level_names):
    M = torch.zeros((num_global, num_level_classes[lvl]), dtype=torch.float32)
    for path_idx, comps in enumerate(split_paths):
        cls = comps[lvl_idx]
        cls_idx = level_to_index[lvl][cls]
        M[path_idx, cls_idx] = 1.0
    M_level[lvl] = M.to(device)



# ==========================================================
# 2. HMCN-LITE ARCHITECTURE with input reuse
# ==========================================================

class HMCN_Lite(nn.Module):
    def __init__(self, num_classes, num_global):
        super().__init__()

        # global tower
        self.G1 = nn.Sequential(
            nn.Linear(768, 384),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.G2 = nn.Sequential(
            nn.Linear(768 + 384, 384),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.G3 = nn.Sequential(
            nn.Linear(768 + 384, 384),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # local heads
        self.local_heads = nn.ModuleDict({
            attr: nn.Linear(384, num_classes[attr])
            for attr in ATTRS
        })

        # global head
        self.global_head = nn.Linear(384, num_global)

        self.dropout = nn.Dropout(0.15)

    def forward(self, x):
        x = self.dropout(x)

        g1 = self.G1(x)
        g2 = self.G2(torch.cat([x, g1], dim=1))
        g3 = self.G3(torch.cat([x, g2], dim=1))

        local_logits = {
            "details_Brand": self.local_heads["details_Brand"](g1),
            "L0_category": self.local_heads["L0_category"](g1),

            "L1_category": self.local_heads["L1_category"](g2),
            "L2_category": self.local_heads["L2_category"](g2),

            "L3_category": self.local_heads["L3_category"](g3),
            "L4_category": self.local_heads["L4_category"](g3),
        }

        global_logits = self.global_head(g3)

        return local_logits, global_logits



## ==========================================================
# FINAL TRAINING + VALIDATION LOOP  (stable + correct)
# ==========================================================

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    total_penalty = 0

    idx = torch.randperm(len(X_train))
    Xs = X_train[idx]
    Ys = {a: Y_train[a][idx] for a in ATTRS_GLOBAL}

    for i in range(0, len(X_train), BATCH):
        xb = Xs[i:i+BATCH]
        local_logits, global_logits = model(xb)

        # ====================================================
        # 1️⃣ Differentiable losses (these train the model)
        # ====================================================
        # Local CE losses
        local_loss = sum(
            criterion(local_logits[a], Ys[a][i:i+BATCH])
            for a in ATTRS
        )

        # Global CE loss
        g_loss = criterion(global_logits, Ys["global_path"][i:i+BATCH])

        # Combine differentiable losses
        loss = local_loss + g_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track differentiable loss
        total_loss += loss.item()

        # ====================================================
        # 2️⃣ NON-differentiable hierarchical violation METRIC
        # ====================================================
        penalty_count = 0
        batch_eff = local_logits["L0_category"].shape[0]

        preds_local_idx = {a: local_logits[a].argmax(1).cpu().numpy()
                           for a in ATTRS}

        parent_child_pairs = [
            ("L0_category", "L1_category"),
            ("L1_category", "L2_category"),
            ("L2_category", "L3_category"),
            ("L3_category", "L4_category"),
        ]

        for parent_attr, child_attr in parent_child_pairs:
            parent_preds = preds_local_idx[parent_attr]
            child_preds  = preds_local_idx[child_attr]

            parent_names = encoders[parent_attr].inverse_transform(parent_preds)
            child_names  = encoders[child_attr].inverse_transform(child_preds)

            for pn, cn in zip(parent_names, child_names):
                allowed = parent_to_child[parent_attr].get(pn, [])
                if cn not in allowed:
                    penalty_count += 1

        # Metric only — NOT used for backprop
        penalty_value = penalty_count / batch_eff
        total_penalty += penalty_value

    print(f"\nEpoch {epoch+1}/{EPOCHS} | "
          f"Loss={total_loss:.2f} | Penalty={total_penalty:.2f}")

# ======================================================
# FINAL VALIDATION LOOP (no length mismatches ever again)
# ======================================================

model.eval()
preds = {a: [] for a in ATTRS_GLOBAL}
trues = {a: [] for a in ATTRS_GLOBAL}

with torch.no_grad():
    for i in range(0, len(X_val), BATCH):

        xb = X_val[i:i+BATCH]
        batch_size_eff = xb.shape[0]

        local_logits, global_logits = model(xb)

        # global path probs
        global_probs = torch.softmax(global_logits, dim=1)

        # 1️⃣ Brand (local only)
        brand_probs = torch.softmax(local_logits["details_Brand"], dim=1)
        preds["details_Brand"].extend(brand_probs.argmax(1).cpu().tolist())

        tb = Y_val["details_Brand"][i:i+batch_size_eff]
        tb = tb.cpu().tolist() if torch.is_tensor(tb) else tb
        trues["details_Brand"].extend(tb)

        # 2️⃣ Each level L0..L4
        for lvl in ["L0_category", "L1_category", "L2_category", "L3_category", "L4_category"]:

            local_probs = torch.softmax(local_logits[lvl], dim=1)

            # aggregated global → level
            M = M_level[lvl]
            global_level_probs = global_probs @ M

            fused = β * local_probs + (1 - β) * global_level_probs

            preds[lvl].extend(fused.argmax(1).cpu().tolist())

            tb = Y_val[lvl][i:i+batch_size_eff]
            tb = tb.cpu().tolist() if torch.is_tensor(tb) else tb
            trues[lvl].extend(tb)

        # 3️⃣ Global path
        preds["global_path"].extend(global_logits.argmax(1).cpu().tolist())

        tb = Y_val["global_path"][i:i+batch_size_eff]
        tb = tb.cpu().tolist() if torch.is_tensor(tb) else tb
        trues["global_path"].extend(tb)


# ======================================================
# METRICS SAFE NOW
# ======================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\nValidation Metrics:")
for a in ATTRS_GLOBAL:

    y_true = trues[a]
    y_pred = preds[a]

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n--- {a} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")




Epoch 1/10 | Loss=8456.35 | Penalty=1041.02

Epoch 2/10 | Loss=8074.56 | Penalty=1016.87

Epoch 3/10 | Loss=7751.79 | Penalty=993.51

Epoch 4/10 | Loss=7450.39 | Penalty=973.67

Epoch 5/10 | Loss=7215.10 | Penalty=958.11

Epoch 6/10 | Loss=6993.97 | Penalty=937.62

Epoch 7/10 | Loss=6805.50 | Penalty=924.04

Epoch 8/10 | Loss=6632.51 | Penalty=912.03

Epoch 9/10 | Loss=6478.99 | Penalty=896.74

Epoch 10/10 | Loss=6344.55 | Penalty=888.35

Validation Metrics:

--- details_Brand ---
Accuracy : 0.0011
Precision: 0.0011
Recall   : 0.0011
F1-score : 0.0011

--- L0_category ---
Accuracy : 0.1710
Precision: 0.1682
Recall   : 0.1710
F1-score : 0.1696

--- L1_category ---
Accuracy : 0.0531
Precision: 0.0519
Recall   : 0.0531
F1-score : 0.0525

--- L2_category ---
Accuracy : 0.0102
Precision: 0.0100
Recall   : 0.0102
F1-score : 0.0101

--- L3_category ---
Accuracy : 0.0124
Precision: 0.0126
Recall   : 0.0124
F1-score : 0.0125

--- L4_category ---
Accuracy : 0.2739
Precision: 0.2604
Recall   : 0

In [ ]:
# ============================================================
# FINAL CLEAN DATA PIPELINE  (NO MISALIGNMENT POSSIBLE)
# ============================================================

import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ------------------------------------------------------------
# 1. Load cleaned training CSV  (this MUST match embeddings)
# ------------------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/phase1_train_clean.csv")

print("Loaded rows:", len(df))

# ------------------------------------------------------------
# 2. Load BERT embeddings saved earlier
# MUST match DF ORDER EXACTLY
# ------------------------------------------------------------
data = torch.load("/content/drive/MyDrive/phase1_embeddings.pt", weights_only=False)

X = data["embeddings"]          # shape (N, 768) — MUST match len(df)
text_list = data["text"]        # debug
labels_raw = data["labels"]     # raw non-numeric columns

print("Embeddings shape:", X.shape)

assert X.shape[0] == len(df), "ERROR: embeddings and dataframe row counts DO NOT match!"

# ------------------------------------------------------------
# 3. Define attribute names
# ------------------------------------------------------------
ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category",
]

# ------------------------------------------------------------
# 4. Fit LabelEncoders ONCE using full df (not per split)
# ------------------------------------------------------------
encoders = {}
Y_np = {}

for a in ATTRS:
    le = LabelEncoder()
    values = df[a].astype(str).tolist()      # MUST use df, not labels_raw
    Y_np[a] = le.fit_transform(values)
    encoders[a] = le

# ------------------------------------------------------------
# 5. Create global_path label
# ------------------------------------------------------------
df["full_path"] = df[["L0_category","L1_category","L2_category","L3_category","L4_category"]]\
                      .astype(str).agg(" || ".join, axis=1)

path_encoder = LabelEncoder()
Y_np["global_path"] = path_encoder.fit_transform(df["full_path"].astype(str))

# full label list
ATTRS_GLOBAL = ATTRS + ["global_path"]

print("Global paths:", len(path_encoder.classes_))

# ------------------------------------------------------------
# 6. Convert all labels to tensors
# ------------------------------------------------------------
Y = {a: torch.tensor(Y_np[a], dtype=torch.long) for a in ATTRS_GLOBAL}

# ------------------------------------------------------------
# 7. SAFE TRAIN–VAL SPLIT  (NO SHUFFLE MISMATCH)
# ------------------------------------------------------------
idx = np.arange(len(X))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

# apply SAME indexing to embeddings and labels
X_train = X[train_idx].to(device)
X_val   = X[val_idx].to(device)

Y_train = {a: Y[a][train_idx].to(device) for a in ATTRS_GLOBAL}
Y_val   = {a: Y[a][val_idx]  .to(device) for a in ATTRS_GLOBAL}

print("Train size:", len(X_train))
print("Val size:", len(X_val))

# ------------------------------------------------------------
# 8. Create num_classes dictionary for each head
# ------------------------------------------------------------
num_classes = {a: len(encoders[a].classes_) for a in ATTRS}
num_global = len(path_encoder.classes_)

print("num_classes:", num_classes)
print("num_global:", num_global)

# ------------------------------------------------------------
# 9. DEBUG CHECKS (IMPORTANT)
# ------------------------------------------------------------

print("\n=== FIRST 10 LABELS PER LEVEL (Should be consistent) ===")
for a in ATTRS_GLOBAL:
    print(a, Y[a][:10].tolist())

print("\n=== FIRST 3 EMBEDDINGS ROWS (Should match df rows) ===")
print(X[:3])


Device: cuda
Loaded rows: 443499
Embeddings shape: torch.Size([443499, 768])
Global paths: 1996
Train size: 376974
Val size: 66525
num_classes: {'details_Brand': 5066, 'L0_category': 27, 'L1_category': 163, 'L2_category': 612, 'L3_category': 1252, 'L4_category': 962}
num_global: 1996

=== FIRST 10 LABELS PER LEVEL (Should be consistent) ===
details_Brand [1455, 3951, 1388, 557, 906, 138, 3361, 4554, 4119, 2704]
L0_category [9, 25, 25, 1, 0, 19, 26, 1, 6, 6]
L1_category [72, 142, 142, 125, 23, 93, 71, 67, 149, 22]
L2_category [509, 530, 530, 474, 397, 319, 588, 517, 4, 131]
L3_category [880, 458, 76, 964, 795, 647, 896, 1055, 865, 611]
L4_category [653, 667, 59, 961, 154, 530, 697, 949, 125, 578]
global_path [1127, 1769, 1761, 458, 21, 1321, 1900, 160, 801, 717]

=== FIRST 3 EMBEDDINGS ROWS (Should match df rows) ===
tensor([[-0.2825, -0.1095, -0.0290,  ..., -0.0858,  0.0283,  0.2782],
        [-0.1180, -0.0496, -0.1738,  ..., -0.2372,  0.2773,  0.3580],
        [-0.3008,  0.1012, -0.36

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


# ==========================================================
# 1. Multi-Task + Global Head Architecture
# ==========================================================

class GECEClassifier(nn.Module):
    def __init__(self, num_classes, num_global):
        super().__init__()

        # Shared representation (simple MLP on top of BERT embeddings)
        self.shared = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Local heads (Brand + L0–L4)
        self.local_heads = nn.ModuleDict({
            attr: nn.Linear(256, num_classes[attr])
            for attr in ATTRS
        })

        # Global head (full path)
        self.global_head = nn.Linear(256, num_global)

    def forward(self, x):
        h = self.shared(x)

        local_logits = {a: self.local_heads[a](h) for a in ATTRS}
        global_logits = self.global_head(h)

        return local_logits, global_logits


# ==========================================================
# 2. Initialize model, loss, optimizer
# ==========================================================

model = GECEClassifier(num_classes, num_global).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

BATCH = 256
EPOCHS = 30


# ==========================================================
# 3. TRAINING LOOP (GE + CE)
# ==========================================================

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    idx = torch.randperm(len(X_train))
    Xs = X_train[idx]
    Ys = {a: Y_train[a][idx] for a in ATTRS_GLOBAL}

    for i in range(0, len(X_train), BATCH):
        xb = Xs[i:i+BATCH]

        local_logits, global_logits = model(xb)

        # -------------------------
        # Local CE losses
        # -------------------------
        local_loss = sum(
            criterion(local_logits[a], Ys[a][i:i+BATCH])
            for a in ATTRS
        )

        # -------------------------
        # Global CE loss
        # -------------------------
        g_loss = criterion(global_logits, Ys["global_path"][i:i+BATCH])

        # -------------------------
        # Total differentiable loss
        # -------------------------
        loss = local_loss + g_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"\nEpoch {epoch+1}/{EPOCHS} | Train Loss={total_loss:.2f}")



    # ======================================================
    # 4. VALIDATION (simple & correct)
    # ======================================================
    model.eval()
    preds = {a: [] for a in ATTRS_GLOBAL}
    trues = {a: [] for a in ATTRS_GLOBAL}

    with torch.no_grad():
        for i in range(0, len(X_val), BATCH):
            xb = X_val[i:i+BATCH]
            batch_eff = xb.size(0)

            local_logits, global_logits = model(xb)

            # Local predictions
            for a in ATTRS:
                preds[a].extend(local_logits[a].argmax(1).cpu().tolist())

                tb = Y_val[a][i:i+batch_eff]
                if torch.is_tensor(tb):
                    tb = tb.cpu().tolist()
                trues[a].extend(tb)

            # Global prediction
            preds["global_path"].extend(global_logits.argmax(1).cpu().tolist())

            tb = Y_val["global_path"][i:i+batch_eff]
            if torch.is_tensor(tb):
                tb = tb.cpu().tolist()
            trues["global_path"].extend(tb)

    print("Validation Metrics:")
    for a in ATTRS_GLOBAL:
        y_true = trues[a]
        y_pred = preds[a]

        acc  = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
        f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

        print(f"\n--- {a} ---")
        print(f"Accuracy : {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall   : {rec:.4f}")
        print(f"F1-score : {f1:.4f}")


Device: cuda

Epoch 1/30 | Train Loss=34317.23
Validation Metrics:

--- details_Brand ---
Accuracy : 0.3404
Precision: 0.2625
Recall   : 0.3404
F1-score : 0.2543

--- L0_category ---
Accuracy : 0.7536
Precision: 0.7451
Recall   : 0.7536
F1-score : 0.7395

--- L1_category ---
Accuracy : 0.6100
Precision: 0.5842
Recall   : 0.6100
F1-score : 0.5781

--- L2_category ---
Accuracy : 0.4933
Precision: 0.4611
Recall   : 0.4933
F1-score : 0.4412

--- L3_category ---
Accuracy : 0.4052
Precision: 0.3605
Recall   : 0.4052
F1-score : 0.3421

--- L4_category ---
Accuracy : 0.5903
Precision: 0.4661
Recall   : 0.5903
F1-score : 0.4840

--- global_path ---
Accuracy : 0.3713
Precision: 0.3068
Recall   : 0.3713
F1-score : 0.3004

Epoch 2/30 | Train Loss=19939.79
Validation Metrics:

--- details_Brand ---
Accuracy : 0.5905
Precision: 0.5676
Recall   : 0.5905
F1-score : 0.5403

--- L0_category ---
Accuracy : 0.7892
Precision: 0.7858
Recall   : 0.7892
F1-score : 0.7796

--- L1_category ---
Accuracy : 0.6848

In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ==========================================================
# 0. LOAD CSVs
# ==========================================================

train_csv_path = "/content/drive/MyDrive/phase1_train_clean.csv"
val_csv_path   = "/content/drive/MyDrive/phase1_val_clean(1).csv"

df_train = pd.read_csv(train_csv_path)
df_val   = pd.read_csv(val_csv_path)

print("Train size:", len(df_train))
print("Val size  :", len(df_val))


# ==========================================================
# 1. LOAD PRECOMPUTED EMBEDDINGS
# ==========================================================

train_emb_path = "/content/drive/MyDrive/phase1_embeddings.pt"
val_emb_path   = "/content/drive/MyDrive/phase1_val_embeddings.pt"

train_data = torch.load(train_emb_path, weights_only=False)
val_data   = torch.load(val_emb_path, weights_only=False)

X_train = train_data["embeddings"]
X_val   = val_data["embeddings"]

print("Train embeddings:", X_train.shape)
print("Val embeddings  :", X_val.shape)

# Make tensors on GPU later
X_train = X_train.float()
X_val   = X_val.float()


# ==========================================================
# 2. LABEL ATTRIBUTES
# ==========================================================

ATTRS = [
    "details_Brand",
    "L0_category",
    "L1_category",
    "L2_category",
    "L3_category",
    "L4_category"
]

ATTRS_GLOBAL = ATTRS + ["global_path"]


# ==========================================================
# 3. BUILD FULL PATH FOR GLOBAL CLASSIFICATION
# ==========================================================

df_train["global_path"] = (
    df_train["L0_category"] + "||" +
    df_train["L1_category"] + "||" +
    df_train["L2_category"] + "||" +
    df_train["L3_category"] + "||" +
    df_train["L4_category"]
)

df_val["global_path"] = (
    df_val["L0_category"] + "||" +
    df_val["L1_category"] + "||" +
    df_val["L2_category"] + "||" +
    df_val["L3_category"] + "||" +
    df_val["L4_category"]
)


# ==========================================================
# 4. FIT LABEL ENCODERS (TRAIN ONLY) AND TRANSFORM VAL
# ==========================================================

encoders = {}
Y_train = {}
Y_val = {}

for a in ATTRS_GLOBAL:
    le = LabelEncoder()
    Y_train[a] = le.fit_transform(df_train[a])
    Y_val[a]   = le.transform(df_val[a])  # IMPORTANT: NO FIT ON VAL
    encoders[a] = le


# Convert to tensors
Y_train = {a: torch.tensor(Y_train[a], dtype=torch.long) for a in ATTRS_GLOBAL}
Y_val   = {a: torch.tensor(Y_val[a],   dtype=torch.long) for a in ATTRS_GLOBAL}


# ==========================================================
# 5. BUILD HIERARCHY MAPPING: parent_to_child
# ==========================================================

parent_to_child = {
    "L0_category": {},
    "L1_category": {},
    "L2_category": {},
    "L3_category": {}
}

for _, row in df_train.iterrows():
    # L0 → L1
    parent = row["L0_category"]
    child  = row["L1_category"]
    parent_to_child["L0_category"].setdefault(parent, set()).add(child)

    # L1 → L2
    parent = row["L1_category"]
    child  = row["L2_category"]
    parent_to_child["L1_category"].setdefault(parent, set()).add(child)

    # L2 → L3
    parent = row["L2_category"]
    child  = row["L3_category"]
    parent_to_child["L2_category"].setdefault(parent, set()).add(child)

    # L3 → L4
    parent = row["L3_category"]
    child  = row["L4_category"]
    parent_to_child["L3_category"].setdefault(parent, set()).add(child)

# convert sets → lists
for k in parent_to_child:
    for p in parent_to_child[k]:
        parent_to_child[k][p] = list(parent_to_child[k][p])


# ==========================================================
# 6. NUMBER OF CLASSES
# ==========================================================

num_classes = {a: len(encoders[a].classes_) for a in ATTRS}
num_global = len(encoders["global_path"].classes_)

print("num_classes:", num_classes)
print("num_global:", num_global)


# ==========================================================
# 7. MOVE EMBEDDINGS TO DEVICE LATER IN TRAINING
# ==========================================================

print("\nData loading complete.")


Train size: 443499
Val size  : 95035
Train embeddings: torch.Size([443499, 768])
Val embeddings  : torch.Size([95035, 768])
num_classes: {'details_Brand': 5066, 'L0_category': 27, 'L1_category': 163, 'L2_category': 612, 'L3_category': 1252, 'L4_category': 962}
num_global: 1996

Data loading complete.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from collections import defaultdict

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --- assume: X_train, X_val, Y_train, Y_val, encoders, parent_to_child, num_classes, num_global, ATTRS, ATTRS_GLOBAL are defined ---
# Move embeddings to device (if not already)
X_train = X_train.to(device)
X_val   = X_val.to(device)

# ==========================================================
# GE + CE Model (same as before)
# ==========================================================
class GECEClassifier(nn.Module):
    def __init__(self, num_classes, num_global):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.local_heads = nn.ModuleDict({
            attr: nn.Linear(256, num_classes[attr])
            for attr in ATTRS
        })
        self.global_head = nn.Linear(256, num_global)

    def forward(self, x):
        h = self.shared(x)
        local_logits = {a: self.local_heads[a](h) for a in ATTRS}
        global_logits = self.global_head(h)
        return local_logits, global_logits

model = GECEClassifier(num_classes, num_global).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

BATCH = 256
EPOCHS = 30

# ==========================================================
# Precompute parent->child mask matrices (for fast batched masking)
# ==========================================================
# For levels: L0 -> L1, L1 -> L2, L2 -> L3, L3 -> L4
parent_levels = ["L0_category", "L1_category", "L2_category", "L3_category"]
child_levels  = ["L1_category", "L2_category", "L3_category", "L4_category"]

parent_child_mask = {}  # key: child_level -> tensor (num_parent_classes, num_child_classes)
for p_lvl, c_lvl in zip(parent_levels, child_levels):
    n_parent = num_classes[p_lvl]
    n_child = num_classes[c_lvl]
    mask = torch.zeros((n_parent, n_child), dtype=torch.float32)  # CPU first
    # parent_to_child uses names; convert names to indices for fast construction
    for parent_name, child_list in parent_to_child[p_lvl].items():
        try:
            parent_idx = encoders[p_lvl].transform([parent_name])[0]
        except Exception:
            continue
        if len(child_list) == 0:
            continue
        child_idxes = encoders[c_lvl].transform(child_list)
        mask[parent_idx, child_idxes] = 1.0
    parent_child_mask[c_lvl] = mask.to(device)  # move to GPU

# ==========================================================
# Fast vectorized cascaded inference (batchwise)
# ==========================================================
def batch_cascaded_preds(local_logits_batch, encoders, parent_child_mask):
    """
    local_logits_batch: dict(level -> tensor (batch, n_classes_level))
    returns: dict(level -> list of predicted indices) for ATTRS, and global predictions are handled elsewhere
    Soft fallback: if parent mask is all zeros for a sample, fallback to argmax of local_logits_batch[level].
    """
    batch_size = next(iter(local_logits_batch.values())).size(0)
    preds = {}

    # details_Brand: local argmax (independent)
    preds["details_Brand"] = local_logits_batch["details_Brand"].argmax(dim=1).cpu().tolist()

    # L0: direct argmax
    preds["L0_category"] = local_logits_batch["L0_category"].argmax(dim=1).cpu().tolist()

    # We'll need parent indices for each sample per level (use L0->L1, L1->L2, ...)
    # Start with parent_idx arrays
    parent_idx_L0 = torch.tensor(preds["L0_category"], dtype=torch.long, device=device)  # (batch,)

    # For each child level do masked selection
    # L1:
    logits_L1 = local_logits_batch["L1_category"]  # (batch, n_L1)
    mask_matrix = parent_child_mask["L1_category"]  # (n_parent_L0, n_child_L1)
    # gather per-sample mask rows: mask_matrix[parent_idx_L0] -> (batch, n_child_L1)
    per_sample_mask = mask_matrix[parent_idx_L0]  # (batch, n_child_L1)
    # compute masked logits: set invalid to large negative
    masked_logits = logits_L1 + (per_sample_mask + 1e-6).log()  # valid=log(1+eps)=~0, invalid=log(eps)=very negative
    # but log trick needs nonzero mask; safer: use masked_probs computed from softmax:
    probs_L1 = F.softmax(logits_L1, dim=1)
    masked_probs_L1 = probs_L1 * per_sample_mask
    # fallback where per_sample_mask.sum(dim=1)==0
    zero_mask = (per_sample_mask.sum(dim=1) == 0)  # bool (batch,)
    # choose indices
    chosen_L1 = torch.zeros(batch_size, dtype=torch.long, device=device)
    # where mask has at least one valid child: pick argmax of masked_probs_L1
    if (~zero_mask).any():
        idxs = (~zero_mask).nonzero(as_tuple=False).squeeze(1)
        part = masked_probs_L1[idxs]
        chosen_L1[idxs] = part.argmax(dim=1)
    # where mask empty: argmax of logits_L1
    if zero_mask.any():
        idxs0 = zero_mask.nonzero(as_tuple=False).squeeze(1)
        part0 = logits_L1[idxs0]
        chosen_L1[idxs0] = part0.argmax(dim=1)
    preds["L1_category"] = chosen_L1.cpu().tolist()

    # L2: parent is L1
    parent_idx_L1 = chosen_L1
    logits_L2 = local_logits_batch["L2_category"]
    mask_matrix = parent_child_mask["L2_category"]  # (n_parent_L1, n_child_L2)
    per_sample_mask = mask_matrix[parent_idx_L1]
    probs_L2 = F.softmax(logits_L2, dim=1)
    masked_probs_L2 = probs_L2 * per_sample_mask
    zero_mask = (per_sample_mask.sum(dim=1) == 0)
    chosen_L2 = torch.zeros(batch_size, dtype=torch.long, device=device)
    if (~zero_mask).any():
        idxs = (~zero_mask).nonzero(as_tuple=False).squeeze(1)
        chosen_L2[idxs] = masked_probs_L2[idxs].argmax(dim=1)
    if zero_mask.any():
        idxs0 = zero_mask.nonzero(as_tuple=False).squeeze(1)
        chosen_L2[idxs0] = logits_L2[idxs0].argmax(dim=1)
    preds["L2_category"] = chosen_L2.cpu().tolist()

    # L3: parent is L2
    parent_idx_L2 = chosen_L2
    logits_L3 = local_logits_batch["L3_category"]
    mask_matrix = parent_child_mask["L3_category"]
    per_sample_mask = mask_matrix[parent_idx_L2]
    probs_L3 = F.softmax(logits_L3, dim=1)
    masked_probs_L3 = probs_L3 * per_sample_mask
    zero_mask = (per_sample_mask.sum(dim=1) == 0)
    chosen_L3 = torch.zeros(batch_size, dtype=torch.long, device=device)
    if (~zero_mask).any():
        idxs = (~zero_mask).nonzero(as_tuple=False).squeeze(1)
        chosen_L3[idxs] = masked_probs_L3[idxs].argmax(dim=1)
    if zero_mask.any():
        idxs0 = zero_mask.nonzero(as_tuple=False).squeeze(1)
        chosen_L3[idxs0] = logits_L3[idxs0].argmax(dim=1)
    preds["L3_category"] = chosen_L3.cpu().tolist()

    # L4: parent is L3
    parent_idx_L3 = chosen_L3
    logits_L4 = local_logits_batch["L4_category"]
    mask_matrix = parent_child_mask["L4_category"] if "L4_category" in parent_child_mask else None
    # Note: parent_child_mask keys created earlier cover L1..L4; if L4 missing, fallback to argmax
    if mask_matrix is not None:
        per_sample_mask = mask_matrix[parent_idx_L3]
        probs_L4 = F.softmax(logits_L4, dim=1)
        masked_probs_L4 = probs_L4 * per_sample_mask
        zero_mask = (per_sample_mask.sum(dim=1) == 0)
        chosen_L4 = torch.zeros(batch_size, dtype=torch.long, device=device)
        if (~zero_mask).any():
            idxs = (~zero_mask).nonzero(as_tuple=False).squeeze(1)
            chosen_L4[idxs] = masked_probs_L4[idxs].argmax(dim=1)
        if zero_mask.any():
            idxs0 = zero_mask.nonzero(as_tuple=False).squeeze(1)
            chosen_L4[idxs0] = logits_L4[idxs0].argmax(dim=1)
    else:
        chosen_L4 = logits_L4.argmax(dim=1)
    preds["L4_category"] = chosen_L4.cpu().tolist()

    return preds


# ==========================================================
# TRAINING LOOP with fast batched cascading in validation
# ==========================================================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    idx = torch.randperm(len(X_train))   # CPU

    Xs = X_train[idx]
    Ys = {a: Y_train[a][idx] for a in ATTRS_GLOBAL}

    for i in range(0, len(X_train), BATCH):
        xb = Xs[i:i+BATCH]                       # already on device
        local_logits, global_logits = model(xb)

        # compute losses (move label slices to device)
        local_loss = sum(criterion(local_logits[a], Ys[a][i:i+xb.size(0)].to(device)) for a in ATTRS)
        g_loss = criterion(global_logits, Ys["global_path"][i:i+xb.size(0)].to(device))
        loss = local_loss + g_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    total_loss = total_loss / len(X_train)
    print(f"\nEpoch {epoch+1}/{EPOCHS} | Train Loss={total_loss:.4f}")

    # Validation
    model.eval()
    preds = {a: [] for a in ATTRS_GLOBAL}
    trues = {a: [] for a in ATTRS_GLOBAL}

    with torch.no_grad():
        for i in range(0, len(X_val), BATCH):
            xb = X_val[i:i+BATCH]  # on device
            batch_eff = xb.size(0)

            # single forward per batch
            local_logits, global_logits = model(xb)

            # compute cascaded local predictions (vectorized)
            batch_local_logits = {a: local_logits[a] for a in ATTRS}
            casc_preds = batch_cascaded_preds(batch_local_logits, encoders, parent_child_mask)

            # gather predictions
            for a in ATTRS:
                preds[a].extend(casc_preds[a])
                # truths from Y_val: move to cpu list
                trues[a].extend(Y_val[a][i:i+batch_eff].cpu().tolist())

            # global path predictions from global_logits
            preds["global_path"].extend(global_logits.argmax(dim=1).cpu().tolist())
            trues["global_path"].extend(Y_val["global_path"][i:i+batch_eff].cpu().tolist())

    # Print metrics
    print("\nValidation Metrics:")
    for a in ATTRS_GLOBAL:
        y_true = trues[a]
        y_pred = preds[a]

        acc  = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
        f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

        print(f"\n--- {a} ---")
        print(f"Accuracy : {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall   : {rec:.4f}")
        print(f"F1-score : {f1:.4f}")

# Save model
save_path = "/content/drive/MyDrive/final_gece_model.pt"
torch.save(model.state_dict(), save_path)
print("\nModel saved to:", save_path)


Device: cuda

Epoch 1/30 | Train Loss=22.1481

Validation Metrics:

--- details_Brand ---
Accuracy : 0.4092
Precision: 0.3422
Recall   : 0.4092
F1-score : 0.3283

--- L0_category ---
Accuracy : 0.7657
Precision: 0.7584
Recall   : 0.7657
F1-score : 0.7533

--- L1_category ---
Accuracy : 0.6140
Precision: 0.5901
Recall   : 0.6140
F1-score : 0.5865

--- L2_category ---
Accuracy : 0.4769
Precision: 0.4369
Recall   : 0.4769
F1-score : 0.4327

--- L3_category ---
Accuracy : 0.3897
Precision: 0.3421
Recall   : 0.3897
F1-score : 0.3381

--- L4_category ---
Accuracy : 0.5535
Precision: 0.5110
Recall   : 0.5535
F1-score : 0.5142

--- global_path ---
Accuracy : 0.4168
Precision: 0.3670
Recall   : 0.4168
F1-score : 0.3521

Epoch 2/30 | Train Loss=12.4172

Validation Metrics:

--- details_Brand ---
Accuracy : 0.6454
Precision: 0.6310
Recall   : 0.6454
F1-score : 0.6050

--- L0_category ---
Accuracy : 0.7983
Precision: 0.7933
Recall   : 0.7983
F1-score : 0.7907

--- L1_category ---
Accuracy : 0.6731